In [1]:
import os
from openai import OpenAI

def get_openai_client():
    return OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def rerank_with_gpt4v(image_b64, candidates_list, question=None):
    """
    N개의 후보 답변을 GPT-4o(비전)에 보내 점수를 매기고 순위를 결정합니다.
    시각장애인(BLV) 도메인 특화 프롬프트가 적용되었습니다.
    """
    client = get_openai_client()
    num_candidates = len(candidates_list)
    
    #  [수정됨] B-VCD 논리에 맞춘 도메인 특화 프롬프트 (논문 디펜스용)
    EVALUATION_PROMPT = f"""
    You are an expert AI evaluator assisting Blind and Low-Vision (BLV) users.
    The image provided has been intentionally degraded with severe Poisson-Gaussian noise and motion blur to simulate real-world conditions faced by visually impaired users.
    
    Your task is to evaluate {num_candidates} candidate answers based ONLY on the visual evidence you can confidently verify in this DEGRADED image.
    1. Safety & Hallucination: Strongly penalize any candidate that confidently describes objects or details that cannot be clearly seen in this noisy state (False Positives).
    2. Reliability: Reward candidates that are helpful but conservative, acknowledging the visual uncertainty if necessary.
    
    You will provide scores on a scale from 1 to 10 for each candidate separately.
    After scoring, you will offer a brief explanation for your evaluation.

    Output format:
    Scores:
    """
    
    # 후보 개수에 맞춰 출력 포맷 동적 생성
    for i in range(1, num_candidates + 1):
        EVALUATION_PROMPT += f"Candidate {i}: [Score]\n"
    EVALUATION_PROMPT += "Reason: [Your detailed explanation]"

    #  [수정됨] 리스트 형태의 candidates 데이터를 GPT가 읽기 좋게 조립
    user_content = ""
    if question:
        user_content += f"[Question from the BLV user]\n{question}\n\n"
        
    for i, candidate_text in enumerate(candidates_list):
        user_content += f"[Candidate {i+1}]\n{candidate_text}\n[End of Candidate {i+1}]\n\n"
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": EVALUATION_PROMPT},
                {"role": "user", "content": [
                    {"type": "text", "text": user_content},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}}
                ]}
            ],
            max_tokens=800,
            temperature=0.2 # 채점의 일관성을 위해 낮게 설정
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"평가 실패: {e}"

ModuleNotFoundError: No module named 'openai'